# H-RESTBERTa: Hierarchical Structure-Aware API Question Answering

**Novel Approach**: Instead of flat span extraction over the entire API schema, we decompose the prediction into 3 hierarchical levels:
- **Level 1**: Predict ROOT entity (e.g., `users`, `products`, `orders`)
- **Level 2**: Predict MIDDLE structure (e.g., `[*]`, single object)
- **Level 3**: Predict LEAF field (e.g., `email`, `name`, `id`)

Each level is conditioned on previous level's prediction (hierarchical classifier heads on shared CodeBERT encoder).

**Target**: Outperform or match RESTBERTa paper baseline (0.8208 on PM) + bring interpretability + efficiency

**Kaggle Setup**:
- Add dataset: `kailram/restberta-datasets`
- Accelerator: **GPU T4 x2**
- Internet: ON
- Time budget: ~6 hours total

In [ ]:
# Cell 1: Install stable versions
!pip install -q "transformers>=4.44.0,<4.48" "huggingface_hub>=0.25,<0.27" "tokenizers>=0.19.0,<0.20" accelerate>=0.34.0 datasets
print('Installed. Kernel → Restart Session, then run Cell 2 onwards')

In [ ]:
# Cell 2: Imports + Config
import os, json, gc, glob, time, random, re, shutil
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datetime import datetime
from tqdm import tqdm
from collections import defaultdict, Counter
from transformers import (
    AutoTokenizer, AutoModel, RobertaModel,
    TrainingArguments, Trainer,
)

# Paths (Kaggle)
DATA_BASE = '/kaggle/input/datasets/kailram/restberta-datasets'
PM_PATH   = os.path.join(DATA_BASE, 'parameter_matching')
ED_PATH   = os.path.join(DATA_BASE, 'endpoint_discovery')
OUTPUT    = '/kaggle/working'
CKPT_DIR  = f'{OUTPUT}/checkpoints'
RESULTS   = f'{OUTPUT}/results'
for d in [CKPT_DIR, RESULTS]:
    os.makedirs(d, exist_ok=True)

# Model config
BACKBONE_CKPT = 'microsoft/codebert-base'   # Same as paper — fair comparison
MODEL_NAME    = 'h_restberta'

# Hierarchical config
MAX_LEVELS = 3
PAD_TOKEN  = '<PAD>'
UNK_TOKEN  = '<UNK>'
NULL_TOKEN = '<NULL>'   # for unanswerable
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, NULL_TOKEN]

# Training hyperparameters
MAX_LENGTH    = 512
TRAIN_SIZE    = 80_000     # QA pairs for training
EPOCHS        = 3
BATCH_SIZE    = 16
GRAD_ACCUM    = 2          # effective batch 32
LR            = 3e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.06
SEED          = 42

# Loss weights for multi-task
L1_WEIGHT = 1.0
L2_WEIGHT = 1.0
L3_WEIGHT = 1.5   # more weight to final answer

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

# Verify data
for split in ['parameter_matching/train/train.json',
              'parameter_matching/validation/validation.json',
              'endpoint_discovery/train/train.json',
              'endpoint_discovery/validation/validation.json']:
    p = os.path.join(DATA_BASE, split)
    sz = os.path.getsize(p)/1e6 if os.path.exists(p) else 0
    print(f"  {'OK' if sz else 'MISSING':>7}  {split} ({sz:.0f} MB)")

In [ ]:
# Cell 3: XPath parser — converts 'users[*].email' to ['users', '[*]', 'email']

def parse_xpath_to_parts(xpath):
    '''
    Parse XPath-like path into list of parts.
    Examples:
      'users[*].email'      → ['users', '[*]', 'email']
      'users.id'            → ['users', 'id']
      'products[*].price'   → ['products', '[*]', 'price']
      'users.{userId}.get'  → ['users', '{userId}', 'get']
      'data[*].items[*].x'  → ['data', '[*]', 'items', '[*]', 'x']
    '''
    if not xpath: return []
    parts = []
    current = ''
    i = 0
    while i < len(xpath):
        c = xpath[i]
        if c == '.':
            if current:
                parts.append(current)
                current = ''
        elif c == '[':
            if current:
                parts.append(current)
                current = ''
            end = xpath.find(']', i)
            if end == -1:
                current += c
                i += 1
                continue
            parts.append(xpath[i:end+1])  # e.g., '[*]'
            i = end + 1
            continue
        elif c == '{':
            if current:
                parts.append(current)
                current = ''
            end = xpath.find('}', i)
            if end == -1:
                current += c
                i += 1
                continue
            parts.append(xpath[i:end+1])  # e.g., '{userId}'
            i = end + 1
            continue
        else:
            current += c
        i += 1
    if current:
        parts.append(current)
    return parts


def get_hierarchical_labels(xpath, max_levels=MAX_LEVELS):
    '''
    Convert XPath to fixed-length hierarchical labels.
    Returns list of length max_levels, padded with PAD_TOKEN.
    If xpath is None/empty (unanswerable), returns all NULL_TOKEN.
    '''
    if not xpath:
        return [NULL_TOKEN] * max_levels
    parts = parse_xpath_to_parts(xpath)
    if not parts:
        return [NULL_TOKEN] * max_levels
    # Take first max_levels; pad if shorter
    result = parts[:max_levels]
    while len(result) < max_levels:
        result.append(PAD_TOKEN)
    return result


def reconstruct_xpath(parts):
    '''Reconstruct XPath from parts list.'''
    result = ''
    for p in parts:
        if p == PAD_TOKEN or p == NULL_TOKEN:
            break
        if not result:
            result = p
        elif p.startswith('[') or p.startswith('{'):
            result += p
        else:
            result += '.' + p
    return result


# Tests
test_cases = [
    ('users[*].email', ['users', '[*]', 'email']),
    ('users.id', ['users', 'id', PAD_TOKEN]),
    ('data[*].items[*].x', ['data', '[*]', 'items']),  # truncated to 3
    ('users.{userId}.get', ['users', '{userId}', 'get']),
    ('', [NULL_TOKEN, NULL_TOKEN, NULL_TOKEN]),
]
print('XPath parser tests:')
for xpath, expected in test_cases:
    actual = get_hierarchical_labels(xpath)
    match = actual == expected
    print(f"  {'✓' if match else '✗'} '{xpath}' → {actual}")
    if not match:
        print(f"       expected: {expected}")

# Reconstruction tests
print('\nReconstruction tests:')
for xpath, _ in test_cases[:-1]:
    parts = get_hierarchical_labels(xpath)
    reconstructed = reconstruct_xpath(parts)
    # For truncated examples, reconstruction may differ
    print(f"  '{xpath}' → parts={parts} → '{reconstructed}'")

In [ ]:
# Cell 4: Build level vocabularies from training data
# Scan all training answers, collect unique tokens at each level

def read_qa_answers(data_path, split, max_qa=None):
    '''Read only answers (not full data) for vocabulary building.'''
    answers = []
    count = 0
    for fpath in sorted(glob.glob(os.path.join(data_path, split, '*.json'))):
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                for q in rec['questions']:
                    if q['answers']['answer_start']:
                        answers.append(q['answers']['text'][0])
                    else:
                        answers.append(None)  # unanswerable
                    count += 1
                    if max_qa and count >= max_qa:
                        return answers
    return answers


def build_vocabularies(pm_answers, ed_answers=None, min_freq=2):
    '''Build vocabularies for each level.'''
    all_answers = list(pm_answers)
    if ed_answers:
        all_answers.extend(ed_answers)

    level_counters = [Counter() for _ in range(MAX_LEVELS)]
    stats = {'total': 0, 'null': 0, 'parsed': 0}

    for ans in all_answers:
        stats['total'] += 1
        if ans is None:
            stats['null'] += 1
            continue
        parts = get_hierarchical_labels(ans)
        stats['parsed'] += 1
        for level_idx, part in enumerate(parts):
            level_counters[level_idx][part] += 1

    # Build vocabularies (only keep tokens appearing >= min_freq times)
    level_vocabs = []
    for level_idx in range(MAX_LEVELS):
        # Always include special tokens
        tokens = list(SPECIAL_TOKENS)
        # Add tokens above frequency threshold
        for tok, freq in level_counters[level_idx].most_common():
            if freq >= min_freq and tok not in tokens:
                tokens.append(tok)
        level_vocabs.append(tokens)

    token_to_id = [{t: i for i, t in enumerate(v)} for v in level_vocabs]

    print(f'\nVocabulary build stats:')
    print(f'  Total QA pairs scanned: {stats["total"]}')
    print(f'  Unanswerable (NULL): {stats["null"]}')
    print(f'  Parsed (answerable): {stats["parsed"]}')

    for level_idx in range(MAX_LEVELS):
        print(f'\n  Level {level_idx+1} vocab size: {len(level_vocabs[level_idx])}')
        top_10 = level_counters[level_idx].most_common(10)
        print(f'    Top-10: {[(t, c) for t, c in top_10]}')

    return level_vocabs, token_to_id


vocabs_path = f'{OUTPUT}/level_vocabs.json'

if os.path.exists(vocabs_path):
    with open(vocabs_path) as f:
        saved = json.load(f)
    level_vocabs = saved['vocabs']
    token_to_id = [{t: i for i, t in enumerate(v)} for v in level_vocabs]
    print(f'Loaded vocabs from cache')
    for level_idx in range(MAX_LEVELS):
        print(f'  Level {level_idx+1}: {len(level_vocabs[level_idx])} tokens')
else:
    print('Reading PM answers for vocabulary...')
    pm_answers = read_qa_answers(PM_PATH, 'train')
    print(f'  PM: {len(pm_answers)} answers')

    print('Reading ED answers for vocabulary...')
    ed_answers = read_qa_answers(ED_PATH, 'train')
    print(f'  ED: {len(ed_answers)} answers')

    level_vocabs, token_to_id = build_vocabularies(pm_answers, ed_answers, min_freq=2)

    # Save
    with open(vocabs_path, 'w') as f:
        json.dump({'vocabs': level_vocabs}, f)
    print(f'\nSaved: {vocabs_path}')

    del pm_answers, ed_answers
    gc.collect()

# Verify vocab sizes are reasonable
N_LEVEL1 = len(level_vocabs[0])
N_LEVEL2 = len(level_vocabs[1])
N_LEVEL3 = len(level_vocabs[2])
print(f'\nFinal: Level 1={N_LEVEL1}, Level 2={N_LEVEL2}, Level 3={N_LEVEL3}')

In [ ]:
# Cell 5: Sample + tokenize training data with hierarchical labels

def read_qa_pairs_sampled(data_path, split, max_qa):
    '''Reservoir sampling — streams large file, keeps random max_qa.'''
    reservoir = []
    count = 0
    for fpath in sorted(glob.glob(os.path.join(data_path, split, '*.json'))):
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    count += 1
                    item = (ctx, q['question'], q['answers'])
                    if len(reservoir) < max_qa:
                        reservoir.append(item)
                    else:
                        j = random.randint(0, count - 1)
                        if j < max_qa:
                            reservoir[j] = item
    print(f'  Scanned {count}, kept {len(reservoir)}')
    random.shuffle(reservoir)
    return reservoir


def read_qa_pairs_validation(data_path):
    qa = []
    for fp in sorted(glob.glob(os.path.join(data_path, 'validation', '*.json'))):
        with open(fp) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    qa.append((ctx, q['question'], q['answers']))
    return qa


cache_path = f'{OUTPUT}/h_restberta_train.npz'

if os.path.exists(cache_path):
    sz = os.path.getsize(cache_path)/1e6
    print(f'Cache exists ({sz:.0f} MB) — skip')
else:
    print('Sampling training data...')
    pm = read_qa_pairs_sampled(PM_PATH, 'train', TRAIN_SIZE)
    ed = read_qa_pairs_sampled(ED_PATH, 'train', min(8_000, TRAIN_SIZE // 10))
    train_data = pm + ed
    del pm, ed
    random.shuffle(train_data)
    print(f'Total: {len(train_data)} QA pairs')

    print(f'\nLoading tokenizer {BACKBONE_CKPT}...')
    tok = AutoTokenizer.from_pretrained(BACKBONE_CKPT)
    pad_on_right = tok.padding_side == 'right'

    all_input_ids = []
    all_masks = []
    all_l1 = []
    all_l2 = []
    all_l3 = []
    stats = {'kept': 0, 'unk_l1': 0, 'unk_l2': 0, 'unk_l3': 0}

    BATCH = 100
    for b0 in tqdm(range(0, len(train_data), BATCH), desc='Tokenizing'):
        batch = train_data[b0:b0+BATCH]
        ctxs = [x[0] for x in batch]
        ques = [x[1] for x in batch]

        # Tokenize (truncation, no overflow — each QA pair = 1 feature)
        tokenized = tok(
            ques if pad_on_right else ctxs,
            ctxs if pad_on_right else ques,
            truncation='only_second' if pad_on_right else 'only_first',
            max_length=MAX_LENGTH,
            padding='max_length',
        )

        for i in range(len(batch)):
            _, _, answers = batch[i]

            # Get XPath answer (if any)
            if answers['answer_start'] and answers['text']:
                xpath = answers['text'][0]
            else:
                xpath = None

            # Convert to hierarchical labels
            parts = get_hierarchical_labels(xpath)
            l1_tok = parts[0]
            l2_tok = parts[1]
            l3_tok = parts[2]

            # Map to IDs (use UNK if not in vocab)
            def safe_id(token, vocab):
                if token in vocab: return vocab[token]
                stats[f'unk_l{level}'] += 1 if level else 0
                return vocab.get(UNK_TOKEN, 1)

            l1_id = token_to_id[0].get(l1_tok, token_to_id[0][UNK_TOKEN])
            l2_id = token_to_id[1].get(l2_tok, token_to_id[1][UNK_TOKEN])
            l3_id = token_to_id[2].get(l3_tok, token_to_id[2][UNK_TOKEN])

            if l1_tok not in token_to_id[0]: stats['unk_l1'] += 1
            if l2_tok not in token_to_id[1]: stats['unk_l2'] += 1
            if l3_tok not in token_to_id[2]: stats['unk_l3'] += 1

            all_input_ids.append(tokenized['input_ids'][i])
            all_masks.append(tokenized['attention_mask'][i])
            all_l1.append(l1_id)
            all_l2.append(l2_id)
            all_l3.append(l3_id)
            stats['kept'] += 1

        del tokenized, ctxs, ques, batch

    print(f'\nStats: kept={stats["kept"]}, UNK L1={stats["unk_l1"]}, L2={stats["unk_l2"]}, L3={stats["unk_l3"]}')

    # Convert to numpy
    all_input_ids = np.array(all_input_ids, dtype=np.int32)
    all_masks = np.array(all_masks, dtype=np.int32)
    all_l1 = np.array(all_l1, dtype=np.int64)
    all_l2 = np.array(all_l2, dtype=np.int64)
    all_l3 = np.array(all_l3, dtype=np.int64)

    print(f'Shape: ids={all_input_ids.shape}')

    np.savez_compressed(cache_path,
                        input_ids=all_input_ids,
                        masks=all_masks,
                        l1=all_l1, l2=all_l2, l3=all_l3)
    print(f'Saved: {cache_path} ({os.path.getsize(cache_path)/1e6:.0f} MB)')
    del all_input_ids, all_masks, all_l1, all_l2, all_l3, train_data, tok
    gc.collect()

In [ ]:
# Cell 6: H-RESTBERTa model architecture

class HRESTBERTa(nn.Module):
    '''
    Hierarchical RESTBERTa with 3-level classifier heads.
    Each level is conditioned on previous level's embedding.
    '''
    def __init__(self, backbone_ckpt, n_level1, n_level2, n_level3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone_ckpt)
        hidden = self.encoder.config.hidden_size  # 768 for CodeBERT

        self.n_level1 = n_level1
        self.n_level2 = n_level2
        self.n_level3 = n_level3

        self.dropout = nn.Dropout(dropout)

        # Level 1: predict root from CLS embedding
        self.level1_head = nn.Linear(hidden, n_level1)

        # Level 1 token embeddings (for conditioning Level 2)
        self.level1_embed = nn.Embedding(n_level1, hidden)

        # Level 2: predict middle, conditioned on CLS + predicted root
        self.level2_head = nn.Linear(hidden * 2, n_level2)

        # Level 2 token embeddings (for conditioning Level 3)
        self.level2_embed = nn.Embedding(n_level2, hidden)

        # Level 3: predict leaf, conditioned on CLS + root + middle
        self.level3_head = nn.Linear(hidden * 3, n_level3)

    def forward(self, input_ids, attention_mask, level1_ids=None, level2_ids=None):
        '''
        Args:
          input_ids: (B, L)
          attention_mask: (B, L)
          level1_ids: (B,) — ground truth Level 1 (for teacher forcing during training)
          level2_ids: (B,) — ground truth Level 2

        Returns:
          l1_logits: (B, n_level1)
          l2_logits: (B, n_level2)
          l3_logits: (B, n_level3)
        '''
        # Encode
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]  # (B, hidden)
        cls = self.dropout(cls)

        # Level 1: predict from CLS only
        l1_logits = self.level1_head(cls)

        # For Level 2: use ground-truth L1 if training, else predicted
        if level1_ids is None:
            level1_ids = l1_logits.argmax(dim=-1)
        l1_embed = self.level1_embed(level1_ids)  # (B, hidden)

        # Level 2: predict from [CLS, L1_embed]
        l2_input = torch.cat([cls, l1_embed], dim=-1)
        l2_input = self.dropout(l2_input)
        l2_logits = self.level2_head(l2_input)

        # For Level 3: use ground-truth L2 if training, else predicted
        if level2_ids is None:
            level2_ids = l2_logits.argmax(dim=-1)
        l2_embed = self.level2_embed(level2_ids)

        # Level 3: predict from [CLS, L1_embed, L2_embed]
        l3_input = torch.cat([cls, l1_embed, l2_embed], dim=-1)
        l3_input = self.dropout(l3_input)
        l3_logits = self.level3_head(l3_input)

        return l1_logits, l2_logits, l3_logits


# Quick test
print('Initializing H-RESTBERTa...')
model = HRESTBERTa(BACKBONE_CKPT, N_LEVEL1, N_LEVEL2, N_LEVEL3).to(device)
print(f'Total params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')

# Forward pass test
dummy_input = torch.randint(0, 1000, (2, 128)).to(device)
dummy_mask = torch.ones(2, 128).to(device)
with torch.no_grad():
    l1, l2, l3 = model(dummy_input, dummy_mask)
print(f'Output shapes: L1={l1.shape}, L2={l2.shape}, L3={l3.shape}')
del model, dummy_input, dummy_mask, l1, l2, l3
torch.cuda.empty_cache()

In [ ]:
# Cell 7: PyTorch Dataset + multi-task loss

class HRESTBERTaDataset(Dataset):
    def __init__(self, cache_path):
        d = np.load(cache_path)
        self.input_ids = d['input_ids'].astype(np.int64)
        self.masks = d['masks'].astype(np.int64)
        self.l1 = d['l1'].astype(np.int64)
        self.l2 = d['l2'].astype(np.int64)
        self.l3 = d['l3'].astype(np.int64)
        print(f'Loaded {len(self.input_ids)} training samples')
        # Class distribution
        unique_l1 = len(np.unique(self.l1))
        unique_l2 = len(np.unique(self.l2))
        unique_l3 = len(np.unique(self.l3))
        print(f'  Unique L1 labels: {unique_l1}, L2: {unique_l2}, L3: {unique_l3}')

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        return {
            'input_ids': torch.from_numpy(self.input_ids[i].copy()),
            'attention_mask': torch.from_numpy(self.masks[i].copy()),
            'level1_ids': torch.tensor(self.l1[i]),
            'level2_ids': torch.tensor(self.l2[i]),
            'level3_ids': torch.tensor(self.l3[i]),
        }


class HRESTBERTaTrainer(Trainer):
    '''Custom Trainer with multi-task loss.'''
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        l1_labels = inputs.pop('level1_ids')
        l2_labels = inputs.pop('level2_ids')
        l3_labels = inputs.pop('level3_ids')

        # Teacher forcing: use ground truth for conditioning
        l1_logits, l2_logits, l3_logits = model(
            inputs['input_ids'],
            inputs['attention_mask'],
            level1_ids=l1_labels,   # teacher forcing
            level2_ids=l2_labels,
        )

        ce = nn.CrossEntropyLoss()
        l1_loss = ce(l1_logits, l1_labels)
        l2_loss = ce(l2_logits, l2_labels)
        l3_loss = ce(l3_logits, l3_labels)

        total_loss = L1_WEIGHT * l1_loss + L2_WEIGHT * l2_loss + L3_WEIGHT * l3_loss

        if return_outputs:
            return total_loss, (l1_logits, l2_logits, l3_logits)
        return total_loss


print('Dataset + Trainer classes defined.')

In [ ]:
# Cell 8: Train H-RESTBERTa

model_save_path = f'{CKPT_DIR}/{MODEL_NAME}'

if os.path.exists(os.path.join(model_save_path, 'pytorch_model.bin')) or \
   os.path.exists(os.path.join(model_save_path, 'model.safetensors')):
    print(f'SKIP: Model already trained at {model_save_path}')
else:
    train_ds = HRESTBERTaDataset(cache_path)

    print(f'\nInitializing H-RESTBERTa...')
    model = HRESTBERTa(BACKBONE_CKPT, N_LEVEL1, N_LEVEL2, N_LEVEL3).to(device)
    print(f'Total params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

    n_steps = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count()))
    print(f'Steps/epoch: ~{n_steps} | Total: ~{n_steps * EPOCHS}')

    args = TrainingArguments(
        output_dir=f'{OUTPUT}/tmp_{MODEL_NAME}',
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type='cosine',
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        logging_steps=100,
        save_strategy='epoch',
        save_total_limit=1,
        report_to='none',
        dataloader_num_workers=2,
        remove_unused_columns=False,
        ddp_find_unused_parameters=False,
    )

    trainer = HRESTBERTaTrainer(model=model, args=args, train_dataset=train_ds)

    # Forward pass test
    model.train()
    batch = {k: v.unsqueeze(0).to(device) for k, v in train_ds[0].items()}
    l1_lab = batch.pop('level1_ids')
    l2_lab = batch.pop('level2_ids')
    l3_lab = batch.pop('level3_ids')
    with torch.amp.autocast('cuda', dtype=torch.float16):
        l1, l2, l3 = model(batch['input_ids'], batch['attention_mask'], l1_lab, l2_lab)
    ce = nn.CrossEntropyLoss()
    loss = L1_WEIGHT*ce(l1, l1_lab) + L2_WEIGHT*ce(l2, l2_lab) + L3_WEIGHT*ce(l3, l3_lab)
    print(f'Sanity check loss: {loss.item():.4f}')
    del batch, l1_lab, l2_lab, l3_lab, l1, l2, l3, loss
    torch.cuda.empty_cache()

    print('\nStarting training...')
    t0 = time.time()
    trainer.train()
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

    # Save model
    if os.path.exists(model_save_path):
        shutil.rmtree(model_save_path)
    os.makedirs(model_save_path)
    torch.save(model.state_dict(), f'{model_save_path}/pytorch_model.bin')
    AutoTokenizer.from_pretrained(BACKBONE_CKPT).save_pretrained(model_save_path)
    # Save vocabs too
    with open(f'{model_save_path}/level_vocabs.json', 'w') as f:
        json.dump({'vocabs': level_vocabs}, f)
    print(f'Saved to {model_save_path}')

    del model, trainer, train_ds
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 9: Sanity check on validation data

# Load model
print('Loading H-RESTBERTa for evaluation...')
model = HRESTBERTa(BACKBONE_CKPT, N_LEVEL1, N_LEVEL2, N_LEVEL3)
state = torch.load(f'{model_save_path}/pytorch_model.bin', map_location=device)
model.load_state_dict(state)
model.to(device).half().eval()
eval_tok = AutoTokenizer.from_pretrained(model_save_path)
print('Loaded.')


def predict_hierarchical(question, context, top_k_per_level=5):
    '''Predict top-K XPaths using beam search.'''
    inputs = eval_tok(
        question, context,
        truncation='only_second',
        max_length=MAX_LENGTH,
        padding='max_length',
        return_tensors='pt'
    ).to(device)

    # Forward pass WITHOUT teacher forcing
    with torch.no_grad():
        l1_logits, l2_logits, l3_logits = model(
            inputs['input_ids'],
            inputs['attention_mask'],
        )

    # Top-K per level
    l1_probs = torch.softmax(l1_logits[0], dim=-1)
    l2_probs = torch.softmax(l2_logits[0], dim=-1)
    l3_probs = torch.softmax(l3_logits[0], dim=-1)

    top_l1 = torch.topk(l1_probs, top_k_per_level)
    top_l2 = torch.topk(l2_probs, top_k_per_level)
    top_l3 = torch.topk(l3_probs, top_k_per_level)

    # Beam: all combinations, ranked by log-sum probability
    candidates = []
    for i in range(top_k_per_level):
        for j in range(top_k_per_level):
            for k in range(top_k_per_level):
                l1_tok = level_vocabs[0][top_l1.indices[i].item()]
                l2_tok = level_vocabs[1][top_l2.indices[j].item()]
                l3_tok = level_vocabs[2][top_l3.indices[k].item()]
                score = (torch.log(top_l1.values[i]) +
                         torch.log(top_l2.values[j]) +
                         torch.log(top_l3.values[k])).item()
                # Skip NULL at level 1 (that means unanswerable)
                if l1_tok == NULL_TOKEN:
                    xpath = None
                else:
                    xpath = reconstruct_xpath([l1_tok, l2_tok, l3_tok])
                candidates.append((score, xpath))

    # Sort by score descending
    candidates.sort(key=lambda x: -x[0])
    # Deduplicate while preserving order
    seen = set()
    unique_preds = []
    for score, xpath in candidates:
        key = xpath if xpath is not None else '__NULL__'
        if key not in seen:
            seen.add(key)
            unique_preds.append((score, xpath))
    return unique_preds[:10]  # top-10 unique predictions


# Quick sanity on 30 samples
print('\nSanity check on 30 PM validation samples:')
sample_qa = []
with open(f'{PM_PATH}/validation/validation.json') as f:
    for line in f:
        rec = json.loads(line.strip())
        for q in rec['questions']:
            if q['answers']['answer_start']:
                sample_qa.append((rec['context'], q['question'], q['answers']))
        if len(sample_qa) >= 30: break

correct_top1 = 0
correct_top5 = 0
for ctx, q, ans in sample_qa[:30]:
    gt = ans['text'][0].strip()
    preds = predict_hierarchical(q, ctx)
    pred_xpaths = [p[1] for p in preds if p[1] is not None]

    if pred_xpaths and pred_xpaths[0] == gt:
        correct_top1 += 1
    if gt in pred_xpaths[:5]:
        correct_top5 += 1

print(f'  Sanity Acc@1: {correct_top1}/30 = {100*correct_top1/30:.0f}%')
print(f'  Sanity Acc@5: {correct_top5}/30 = {100*correct_top5/30:.0f}%')
print(f'  Target: >= 18/30 (60%) Acc@1 for feasibility')

In [ ]:
# Cell 10: Full evaluation on PM and ED

TOP_K = list(range(1, 11))
N_BEST_PER_LEVEL = 6  # 6^3 = 216 beams per query


def full_evaluation(data_path, ds_name, batch_size=16):
    print(f'\n{"="*60}\n  EVAL: H-RESTBERTa on {ds_name}\n{"="*60}')

    qa_pairs = read_qa_pairs_validation(data_path)
    print(f'  {len(qa_pairs)} QA pairs')

    results = [{'rank': k, 'correct': 0, 'correct_answerable': 0,
                'correct_non_answerable': 0,
                'total': 0, 'total_answerable': 0, 'total_non_answerable': 0}
               for k in TOP_K]

    t0 = time.time()

    for b0 in tqdm(range(0, len(qa_pairs), batch_size), desc='Eval'):
        batch = qa_pairs[b0:b0+batch_size]
        ctxs = [x[0] for x in batch]
        ques = [x[1] for x in batch]
        answers_list = [x[2] for x in batch]

        inputs = eval_tok(
            ques, ctxs,
            truncation='only_second',
            max_length=MAX_LENGTH,
            padding='max_length',
            return_tensors='pt',
        ).to(device)

        with torch.no_grad():
            l1_logits, l2_logits, l3_logits = model(
                inputs['input_ids'], inputs['attention_mask'],
            )

        l1_probs = torch.softmax(l1_logits, dim=-1)
        l2_probs = torch.softmax(l2_logits, dim=-1)
        l3_probs = torch.softmax(l3_logits, dim=-1)

        top_l1 = torch.topk(l1_probs, N_BEST_PER_LEVEL, dim=-1)
        top_l2 = torch.topk(l2_probs, N_BEST_PER_LEVEL, dim=-1)
        top_l3 = torch.topk(l3_probs, N_BEST_PER_LEVEL, dim=-1)

        for bi in range(len(batch)):
            ans = answers_list[bi]
            gt_xpath = ans['text'][0].strip() if ans['answer_start'] and ans['text'] else None
            is_oos = gt_xpath is None

            # Build beam
            candidates = []
            for i in range(N_BEST_PER_LEVEL):
                l1_tok_id = top_l1.indices[bi, i].item()
                l1_tok = level_vocabs[0][l1_tok_id]
                l1_logp = torch.log(top_l1.values[bi, i] + 1e-12).item()
                for j in range(N_BEST_PER_LEVEL):
                    l2_tok_id = top_l2.indices[bi, j].item()
                    l2_tok = level_vocabs[1][l2_tok_id]
                    l2_logp = torch.log(top_l2.values[bi, j] + 1e-12).item()
                    for k in range(N_BEST_PER_LEVEL):
                        l3_tok_id = top_l3.indices[bi, k].item()
                        l3_tok = level_vocabs[2][l3_tok_id]
                        l3_logp = torch.log(top_l3.values[bi, k] + 1e-12).item()
                        score = l1_logp + l2_logp + l3_logp
                        if l1_tok == NULL_TOKEN:
                            xpath = None
                        else:
                            xpath = reconstruct_xpath([l1_tok, l2_tok, l3_tok])
                        candidates.append((score, xpath))
            candidates.sort(key=lambda x: -x[0])

            # Deduplicate
            seen = set()
            pred_ranked = []
            for _, xpath in candidates:
                key = xpath if xpath is not None else '__NULL__'
                if key in seen: continue
                seen.add(key)
                pred_ranked.append(xpath)
                if len(pred_ranked) >= 10: break

            # Find rank of correct
            rank = None
            for ri, pred in enumerate(pred_ranked):
                if is_oos:
                    if pred is None:
                        rank = ri + 1; break
                else:
                    if pred == gt_xpath:
                        rank = ri + 1; break

            for kr in results:
                if rank is not None and rank <= kr['rank']:
                    kr['correct'] += 1
                    if is_oos: kr['correct_non_answerable'] += 1
                    else: kr['correct_answerable'] += 1
                kr['total'] += 1
                if is_oos: kr['total_non_answerable'] += 1
                else: kr['total_answerable'] += 1

    for kr in results:
        t = kr['total']; ta = kr['total_answerable']; tn = kr['total_non_answerable']
        kr['accuracy'] = kr['correct']/t if t else 0
        kr['accuracy_answerable'] = kr['correct_answerable']/ta if ta else 0
        kr['accuracy_non_answerable'] = kr['correct_non_answerable']/tn if tn else 0

    elapsed = time.time() - t0
    print(f'\n  Done: {elapsed/60:.1f} min')
    print(f'  Answerable: {results[0]["total_answerable"]} | Non-ans: {results[0]["total_non_answerable"]}')
    print(f'  K   Acc@K   Ans     Non')
    for r in results:
        print(f"  @{r['rank']:>2}  {r['accuracy']:.4f}  {r['accuracy_answerable']:.4f}  {r['accuracy_non_answerable']:.4f}")

    return {
        'model': MODEL_NAME, 'dataset': ds_name, 'type': 'hierarchical',
        'total_samples': results[0]['total'],
        'total_answerable': results[0]['total_answerable'],
        'total_non_answerable': results[0]['total_non_answerable'],
        'elapsed_seconds': elapsed,
        'results': results,
    }


all_results = []
r = full_evaluation(PM_PATH, 'parameter_matching')
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'{RESULTS}/{MODEL_NAME}_pm_{ts}.json', 'w') as f:
    json.dump(r, f, indent=2)
all_results.append(r)

r = full_evaluation(ED_PATH, 'endpoint_discovery')
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'{RESULTS}/{MODEL_NAME}_ed_{ts}.json', 'w') as f:
    json.dump(r, f, indent=2)
all_results.append(r)

print('\n✓ Evaluation complete!')

In [ ]:
# Cell 11: Per-level accuracy analysis (for ablations/paper)

def per_level_analysis(data_path, ds_name):
    '''Analyze accuracy at each level independently.'''
    print(f'\nPer-level analysis: {ds_name}')
    qa_pairs = read_qa_pairs_validation(data_path)

    l1_correct = l2_correct = l3_correct = joint_correct = total = 0
    batch_size = 16

    for b0 in tqdm(range(0, len(qa_pairs), batch_size), desc='  Analysis'):
        batch = qa_pairs[b0:b0+batch_size]
        ctxs = [x[0] for x in batch]
        ques = [x[1] for x in batch]
        answers_list = [x[2] for x in batch]

        inputs = eval_tok(ques, ctxs, truncation='only_second',
                          max_length=MAX_LENGTH, padding='max_length',
                          return_tensors='pt').to(device)

        with torch.no_grad():
            l1_logits, l2_logits, l3_logits = model(
                inputs['input_ids'], inputs['attention_mask']
            )

        l1_pred = l1_logits.argmax(dim=-1).cpu().numpy()
        l2_pred = l2_logits.argmax(dim=-1).cpu().numpy()
        l3_pred = l3_logits.argmax(dim=-1).cpu().numpy()

        for bi in range(len(batch)):
            ans = answers_list[bi]
            gt_xpath = ans['text'][0].strip() if ans['answer_start'] and ans['text'] else None
            gt_parts = get_hierarchical_labels(gt_xpath)

            gt_l1_id = token_to_id[0].get(gt_parts[0], token_to_id[0][UNK_TOKEN])
            gt_l2_id = token_to_id[1].get(gt_parts[1], token_to_id[1][UNK_TOKEN])
            gt_l3_id = token_to_id[2].get(gt_parts[2], token_to_id[2][UNK_TOKEN])

            l1_hit = (l1_pred[bi] == gt_l1_id)
            l2_hit = (l2_pred[bi] == gt_l2_id)
            l3_hit = (l3_pred[bi] == gt_l3_id)

            if l1_hit: l1_correct += 1
            if l2_hit: l2_correct += 1
            if l3_hit: l3_correct += 1
            if l1_hit and l2_hit and l3_hit: joint_correct += 1
            total += 1

    print(f'\n  Level 1 accuracy: {l1_correct/total:.4f}')
    print(f'  Level 2 accuracy: {l2_correct/total:.4f}')
    print(f'  Level 3 accuracy: {l3_correct/total:.4f}')
    print(f'  Joint (all 3):    {joint_correct/total:.4f}')
    return {
        'level1_acc': l1_correct/total,
        'level2_acc': l2_correct/total,
        'level3_acc': l3_correct/total,
        'joint_acc': joint_correct/total,
    }


pm_levels = per_level_analysis(PM_PATH, 'parameter_matching')
ed_levels = per_level_analysis(ED_PATH, 'endpoint_discovery')

# Save
with open(f'{RESULTS}/{MODEL_NAME}_per_level_analysis.json', 'w') as f:
    json.dump({'pm': pm_levels, 'ed': ed_levels}, f, indent=2)

In [ ]:
# Cell 12: Final comparison table
baseline = [
    {'model': 'CB-PM (paper)', 'dataset': 'parameter_matching', 'type': 'baseline',
     'results': [{'rank': 1, 'accuracy': 0.8195}, {'rank': 3, 'accuracy': 0.9613},
                 {'rank': 5, 'accuracy': 0.9759}, {'rank': 10, 'accuracy': 0.9816}]},
    {'model': 'CB-PM+ED (paper)', 'dataset': 'parameter_matching', 'type': 'baseline',
     'results': [{'rank': 1, 'accuracy': 0.8208}]},
    {'model': 'CB-ED (paper)', 'dataset': 'endpoint_discovery', 'type': 'baseline',
     'results': [{'rank': 1, 'accuracy': 0.8844}, {'rank': 3, 'accuracy': 0.9535},
                 {'rank': 5, 'accuracy': 0.9678}, {'rank': 10, 'accuracy': 0.9749}]},
]

combined = baseline + all_results

print(f'\n{"="*110}')
print(f'H-RESTBERTa vs RESTBERTa paper')
print(f'{"="*110}')
print(f'{"Model":<40} {"Type":<14} {"Dataset":<22} {"Acc@1":>7} {"Acc@3":>7} {"Acc@5":>7} {"Acc@10":>7}')
print('-'*110)
for r in combined:
    acc = {x['rank']: x.get('accuracy', 0) for x in r['results']}
    print(f'{r["model"]:<40} {r["type"]:<14} {r["dataset"]:<22} '
          f'{acc.get(1,0):>7.4f} {acc.get(3,0):>7.4f} {acc.get(5,0):>7.4f} {acc.get(10,0):>7.4f}')
print('='*110)

for ds in ['parameter_matching', 'endpoint_discovery']:
    paper = max([{x['rank']: x['accuracy'] for x in r['results']}.get(1, 0)
                 for r in baseline if r['dataset'] == ds] or [0])
    ours = max([{x['rank']: x['accuracy'] for x in r['results']}.get(1, 0)
                for r in all_results if r['dataset'] == ds] or [0])
    diff = (ours - paper) * 100
    sign = '🎯' if diff > 0 else '⚠'
    print(f'\n{sign} {ds}: H-RESTBERTa {ours:.4f} vs paper {paper:.4f} → {diff:+.2f}%')

with open(f'{RESULTS}/{MODEL_NAME}_final_comparison.json', 'w') as f:
    json.dump(combined, f, indent=2)
print(f'\nSaved: {RESULTS}/{MODEL_NAME}_final_comparison.json')